In [ ]:
# ============================================================
# Notebook 10: "What the Geometry Cannot See"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence

from regression_geometry.core import ColumnSpace, Projection, HatMatrix, Ellipsoid
from regression_geometry.core import frisch_waugh_lovell, angle_between, demean
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard
from regression_geometry.exercises import predict_first, diagnose_first, memo, reveal

# --- Reproducibility ---
np.random.seed(42)
df = load_meridian()

> *"Knowing the shape of what you know is the beginning of knowing what you don't."*

## The Victory Lap

Before we talk about limits, use everything one more time.

In [ ]:
# Choose Your Own Regression — Meridian sandbox
# Uncomment the predictors you want to include.

predictors = [
    'experience',
    'education',
    # 'performance',
    # 'gender',
    # 'job_level',
]

# Which coefficient's Relevant Triangle to show
show_triangle_for = 'experience'

# --- the rest runs automatically ---
X_sandbox = df[predictors].values
y_sandbox = df['salary'].values

cs_sb = ColumnSpace(X_sandbox, add_intercept=True)
proj_sb = cs_sb.project(y_sandbox)

X_sm_sb = sm.add_constant(X_sandbox)
model_sb = sm.OLS(y_sandbox, X_sm_sb).fit()
print(model_sb.summary())

In [ ]:
# Scoreboard, eigenvalues, leverage, and Relevant Triangle
sb = GeometricScoreboard(
    proj=proj_sb, cs=cs_sb,
    active_gauges=['theta', 'r_squared', 'leverage', 'residual_norm', 'kappa'],
    mode='widget' if INTERACTIVE else 'static'
)
sb.display()

fig_eig = viz.plot_eigenvalue_bar(cs_sb, title="Eigenvalue Structure")

hm_sb = HatMatrix(proj_sb.H)
fig_lev = viz.plot_leverage(hm_sb, title="Leverage")

tri_idx = predictors.index(show_triangle_for) + 1
fig_tri = viz.plot_relevant_triangle(
    cs_sb, y_sandbox, tri_idx,
    title=f"Relevant Triangle: {show_triangle_for}"
)

In [ ]:
# 🛑 PREDICT FIRST
predict_first(
    description=(
        "Try adding and removing variables in the cell above. "
        "Before each change, predict: will R² go up or down? "
        "Will the condition number change? Will the coefficient you're watching move?"
    ),
    question=(
        "Pick one variable to add (or remove). Write your prediction for R², "
        "κ, and the coefficient on '" + show_triangle_for + "' before re-running."
    )
)

In [ ]:
# Free exploration — the sandbox is yours


In [ ]:
# Try things. Break things.


## What the Geometry Can See

Here is what the geometric framework gave you — the things that are now *visible* to you that were invisible before this course.

- **Coefficients** are coordinates of a projection — the shadow's location on the wall
- **R²** is the cosine of an angle — how close the shadow is to the object
- **Variance decomposition** is the Pythagorean theorem — squared lengths of a right triangle
- **"Controlling for"** means residualizing — peeling away shared information (FWL)
- **Leverage** is arm length in the column space — how far an observation is from the crowd
- **Influence** is leverage × surprise — who actually moves the shadow
- **Multicollinearity** is a thin column space — the eigenvalue ellipsoid collapsing
- **The F-test** compares two projections — is the gap between them bigger than noise?
- **Ridge** shrinks the unstable directions — selective geometric surgery
- **LASSO** exploits diamond corners — sparsity from geometry

Every one of these was mysterious before. Now each one is a picture you can draw.

## What the Geometry Cannot See

The geometric framework is powerful. But it has a boundary. Three things lie beyond it.

### Limit 1: Causality

The geometry tells you how well you're projecting. It cannot tell you whether you're projecting onto the right wall. Omitted variable bias is invisible to every geometric diagnostic. The Scoreboard shows green for a regression that's perfectly projected onto the wrong subspace.

To choose the right wall, you need a causal model — a theory about what generates the data. Tools for this: directed acyclic graphs (DAGs), potential outcomes, natural experiments, instrumental variables. These are not geometric tools. They are logical and structural tools that tell you WHICH column space to project onto.

### Limit 2: The Inner Product

Everything in this course assumed a Euclidean inner product — all observations weighted equally, all directions in ℝⁿ treated the same. But what if observations have different variances (heteroscedasticity)? Or are correlated (autocorrelation)?

When the inner product is wrong, the "perpendicular" isn't really perpendicular. The "closest point" isn't really closest. The entire geometric framework distorts.

The fix: change the inner product. Generalized Least Squares (GLS) uses a different metric — one that accounts for unequal variances or correlations. The geometry is the same (it's still a projection), but the shape of "perpendicular" changes. OLS geometry is the geometry of ONE specific inner product. GLS is the geometry of a different one.

In [ ]:
# Heteroscedastic data: the fan pattern
rng = np.random.default_rng(42)
x_het = rng.uniform(1, 10, 200)
noise_het = rng.normal(0, 1, 200) * x_het  # variance grows with x
y_het = 3 + 2 * x_het + noise_het

X_het = sm.add_constant(x_het)
model_het = sm.OLS(y_het, X_het).fit()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_het, model_het.resid, alpha=0.5, s=20, color=RESIDUAL)
ax.axhline(0, color=SECONDARY, lw=1, ls='--')
ax.set(xlabel='x', ylabel='Residuals', title='OLS Residuals (fan pattern)')
plt.tight_layout()
plt.show()

print("The fan shape means the variance isn't constant.")
print("OLS assumes equal weights. A different course would show you")
print("the GLS projection \u2014 same theorem, different metric.")

### Limit 3: Beyond Linearity

The column space is flat — a plane, a hyperplane. The projection can only land on a flat surface. If the true relationship is curved, the best flat shadow might be a poor representation.

You saw a hint of this in Notebook 3's overfitting trap: adding polynomial terms expanded the column space to include curved surfaces, and the projection improved — until it didn't (out of sample). The tension between flexibility and stability is a different geometric problem: the geometry of bias-variance tradeoff, kernel methods, and neural network function spaces.

These are real geometric frameworks — but they require different geometry than what this course teaches.

In [ ]:
# 🛑 DIAGNOSE FIRST: time series with autocorrelation
rng_ts = np.random.default_rng(99)
n_ts = 150
t = np.arange(n_ts)

# AR(1) errors with rho = 0.85
eps = np.zeros(n_ts)
for i in range(1, n_ts):
    eps[i] = 0.85 * eps[i-1] + rng_ts.normal(0, 1)
y_ts = 10 + 0.5 * t + eps

X_ts = sm.add_constant(t)
model_ts = sm.OLS(y_ts, X_ts).fit()

cs_ts = ColumnSpace(t.reshape(-1, 1), add_intercept=True)
proj_ts = cs_ts.project(y_ts)
sb_ts = GeometricScoreboard(
    proj=proj_ts, cs=cs_ts,
    active_gauges=['theta', 'r_squared', 'leverage', 'residual_norm', 'kappa'],
    mode='widget' if INTERACTIVE else 'static'
)
sb_ts.display()

dw = float(sm.stats.stattools.durbin_watson(model_ts.resid))
print(f"Durbin-Watson statistic: {dw:.3f}")

diagnose_first(
    summary_text=str(model_ts.summary()),
    question=(
        f"The Scoreboard says this regression is healthy. "
        f"But look at the Durbin-Watson statistic: {dw:.3f}. "
        f"What does this suggest? Is this a problem the geometric framework can diagnose?"
    )
)

The Durbin-Watson statistic flags autocorrelation — the residuals are not independent. This violates the assumption behind our inner product. The geometry we learned assumed spherical errors (equal variance, no correlation). Autocorrelated residuals mean the inner product is wrong — and a different geometry (GLS) is needed.

## Where to Go Next

This course taught you one theorem from ten angles. Here's what comes next — and how each connects to what you already know.

**Causal inference** (DAGs, potential outcomes, IV): choosing the right column space. The geometry tells you how to project; causal inference tells you where to aim.

**GLS / WLS / HAC**: changing the inner product. The projection theorem still holds, but "perpendicular" means something different when observations have unequal reliability.

**Bayesian regression**: coefficients as distributions. The constraint sphere from Ridge becomes a prior. The posterior is what you get when geometry meets belief.

**Time series**: the geometry of autocorrelation. The path structure in ℝⁿ creates dependencies that reshape the inner product.

**High-dimensional regression**: when p > n and the column space fills all of ℝⁿ. Every y can be perfectly fitted — and that's the problem.

**Nonparametric methods**: beyond flat column spaces. Kernel methods, trees, and neural networks define curved surfaces to project onto.

---

### ✍️ The Memo

Write a one-page letter to yourself six months from now. It's Tuesday morning. You've just run a regression and something feels off. What will you do differently now that you can SEE the geometry?

There are no banned words for this memo. Use whatever language — geometric, algebraic, plain English — captures what you've learned. This memo is for you.

---

In [ ]:
# YOUR MEMO:

In [ ]:
# Geometric Scoreboard — one last time, Meridian full model
y_final = df['salary'].values
X_final = df[['experience', 'education', 'performance']].values
cs_final = ColumnSpace(X_final, add_intercept=True)
proj_final = cs_final.project(y_final)

sb_last = GeometricScoreboard(
    proj=proj_final, cs=cs_final,
    active_gauges=['theta', 'r_squared', 'leverage', 'residual_norm', 'kappa'],
    mode='widget' if INTERACTIVE else 'static'
)
sb_last.display()

Five gauges. One theorem. You can read them now.

θ tells you the angle — how far the shadow is from the object. κ tells you the condition — how stable the shadow is on the wall. tr(H)/n tells you the leverage — how much room each observation has to pull. ‖e‖/‖y‖ tells you the residual — how much the wall didn't catch. R² tells you the same thing as θ, from a different angle.

None of them tell you whether it's the right wall. For that, you bring your brain.

---

### Summary

What you learned in this course:

Regression is a projection. Coefficients are shadows. The residual is perpendicular. R² is an angle. Variance decomposition is Pythagoras. Gauss-Markov is "closest point." Controlling for is residualizing. Leverage is arm length. Influence is leverage times surprise. Multicollinearity is a thin column space. The F-test compares two projections. Ridge shrinks the unstable directions. LASSO exploits diamond corners. And the geometry can't tell you which wall to aim at.

One theorem. Ten notebooks. You've been reading regression output your whole career.

Now you see it.

---

---

*Regression from the Inside: Seeing the Geometry of Linear Models*

Built on the Projection Theorem.

The geometric perspective on regression has a long lineage — Seber (1977), Davidson & MacKinnon (1993), Christensen (2011) among others — and this course owes them a debt for showing that the algebra has a shape.

The geometry was always there. You just needed someone to hand you the flashlight.

---